# Разведочный анализ и формирование признаков

Блокнот связан с третьей главой пособия и показывает, как от первичного описания набора данных перейти к содержательным признакам для дальнейшего анализа.

## Цель и ожидаемые результаты

После выполнения блокнота читатель должен:

- рассчитывать базовые описательные статистики;
- визуально и численно проверять структуру данных;
- формировать производные признаки;
- объяснять, почему признак может быть полезен в последующей задаче.

## Постановка задачи

Требуется продемонстрировать, как из ряда наблюдений нагрузки получить набор интерпретируемых признаков, пригодных для аналитики и прогнозирования.

## Используемые библиотеки

| Библиотека | Назначение |
| --- | --- |
| pandas | табличные преобразования и расчет признаков |
| numpy | числовые операции |
| matplotlib | простая визуализация временного хода |
| src.display_utils | компактный вывод итогов и наблюдений |

## Источник и описание данных

В блокноте используется небольшой синтетический временной ряд нагрузки, достаточный для демонстрации описательной статистики, временной структуры и формирования лаговых и календарных признаков.

## Перечень признаков

| Признак | Тип | Смысл |
| --- | --- | --- |
| timestamp | datetime | момент времени |
| load_mw | float | текущая нагрузка |
| temperature_c | float | температура окружающей среды |
| hour | int | час суток |
| load_lag_1 | float | нагрузка на предыдущем шаге |
| load_roll_3 | float | скользящее среднее за три шага |

## Этапы анализа

### 1. Формирование ряда наблюдений
Создается учебный набор с временной шкалой, нагрузкой и температурой.

### 2. Описательная статистика
Рассчитываются базовые характеристики признаков.

### 3. Построение производных признаков
Формируются календарные и лаговые признаки.

### 4. Интерпретация
Определяется, какие из признаков потенциально значимы для моделирования.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.display_utils import display_callout, display_metric_table

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

rng = pd.date_range("2025-02-01 00:00", periods=24, freq="h")
df = pd.DataFrame(
    {
        "timestamp": rng,
        "load_mw": [118, 116, 115, 114, 113, 117, 125, 132, 138, 141, 145, 149, 151, 150, 148, 146, 143, 144, 147, 149, 146, 140, 133, 126],
        "temperature_c": [-18, -18, -19, -19, -20, -19, -17, -16, -14, -13, -11, -9, -8, -7, -7, -8, -9, -10, -11, -12, -13, -15, -16, -17],
    }
)
df.head()

## Интерпретация

Набор содержит часовую динамику нагрузки и температуры. Уже на этом этапе можно предположить наличие суточной структуры и зависимости нагрузки от внешних условий.

In [ ]:
stats = df[["load_mw", "temperature_c"]].describe().T
stats

## Интерпретация

Описательная статистика помогает быстро оценить диапазоны, центральные тенденции и разброс. Эти сведения нужны не только для общего понимания набора данных, но и для контроля выбросов и масштаба признаков.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["load_lag_1"] = df["load_mw"].shift(1)
df["load_roll_3"] = df["load_mw"].rolling(window=3).mean()
df[["timestamp", "load_mw", "hour", "load_lag_1", "load_roll_3"]].head(8)

## Интерпретация

Календарные и лаговые признаки позволяют перевести временную структуру процесса в форму, пригодную для дальнейшего моделирования. При этом важно помнить, что лаговые признаки неизбежно создают пропуски в начале ряда.

In [ ]:
corr = df[["load_mw", "temperature_c", "hour", "load_lag_1", "load_roll_3"]].corr(numeric_only=True)
corr

## Интерпретация

Корреляционная таблица не заменяет полноценный анализ, но помогает увидеть, какие признаки потенциально связаны с целевой величиной. В данном примере особенно интересны лаговая нагрузка и скользящее среднее.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["timestamp"], df["load_mw"], marker="o", label="Нагрузка, МВт")
ax.set_title("Суточный ход нагрузки")
ax.set_xlabel("Время")
ax.set_ylabel("МВт")
ax.grid(True, alpha=0.3)
ax.legend();

## Интерпретация

График подтверждает наличие выраженного суточного профиля. Такой рисунок уже на этапе разведочного анализа подсказывает необходимость учитывать время суток и инерционность процесса.

In [ ]:
display_metric_table(
    {
        "mean_load": float(df["load_mw"].mean()),
        "std_load": float(df["load_mw"].std()),
        "max_load": float(df["load_mw"].max()),
        "min_load": float(df["load_mw"].min()),
    },
    decimals=2,
)

display_callout(
    "Вывод по признакам",
    "Для дальнейшего моделирования целесообразно сохранить временные признаки и лаговые характеристики нагрузки.",
    tone="info",
)

## Итоговые выводы

- Разведочный анализ необходим для выбора осмысленного признакового пространства.
- Производные признаки должны отражать физику или структуру процесса.
- Лаговые и календарные признаки особенно важны для задач временного ряда и прогноза нагрузки.

## Вопросы и задания для самостоятельной работы

1. Добавьте признак "день недели" и оцените его потенциальную полезность.
2. Постройте дополнительный график температуры и сравните его форму с графиком нагрузки.
3. Сформируйте еще один агрегированный признак и обоснуйте его выбор.

## Источники

1. [Глава 3 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/03-eda-and-features.md)
2. [Требования к оформлению учебных блокнотов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/notebook-style-guide.md)